In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab1.ipynb")

---

<h3><center>E178/ME276DS - Statistics and Data Science for Engineers</center></h3>

<h1><center>Lab 1<br></center></h1>

<h2><center>Data exploration and repair with Pandas,<br>Simulating random variables with SciPy.</center></h2>

---

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import rv_discrete
import matplotlib.pyplot as plt
from resources.hashutils import *

# Part 1. Data exploration and repair with Pandas

The pandas package was introduced in lab0. Here we will begin to explore the tools provided in this package for inspecting and repairing datasets. Throughout this exercise you will use a number of pandas methods. Below we provide links to the documentation for the pandas methods needed to complete this lab exercise. 


Package methods:
- [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html): read a CSV file into a DataFrame.
- [`to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html): convert values to numbers.
- [`to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html): convert values to datetime objects.

Series methods:
- [`value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html): count occurrences of each value.
- [`unique`](https://pandas.pydata.org/docs/reference/api/pandas.Series.unique.html): list distinct values in a Series.



DataFrame methods:
- [`info`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html): inspect columns, data types, and missing values.
- [`head`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html): inspect the first rows of a DataFrame.
- [`copy`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html): make an independent copy before editing data.
- [`astype`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html): convert values to a specified data type.
- [`describe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html): calculate descriptive statistics.
- [`isna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html): identify missing values.
- [`loc`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html): select rows and columns by labels or Boolean conditions.
- [`plot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html): create plots directly from pandas objects.
- [`boxplot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.bocplot.html): make a box-and-whisker plot from DataFrame columns, optionally grouped by some other columns. 

## Part 1.1) Load and inspect the dataset

We begin the exercise similarly to lab0, by loading and checking the high-level properties of the dataset: how many rows and columns it has, the data types assigned to each column, and performing a visual inspection of the table. 

Our dataset records sales transactions for a retail company. Each transaction has a purchase date, store location, product category, price, as well as information about the customer's age and their satisfaction. The dataset is not perfect. It has problems including inappropriate data types, inconsistent text labels, and others. We will use pandas to find these problems and fix them. 

**Instructions**

1. Load `retail_sales.csv` from the `resources` folder into a DataFrame named `raw_data` using `pd.read_csv`. 
2. Save the number of rows and columns respectively to variables named `N` and `D`.
3. Use the  `info()` method to inspect the structure of the dataset. Do you see any rows with null entries? 
4. Use `head(20)` to view the first 20 rows of the table. Visually scan the displayed values for clues about fields that may need some repair. 

In [ ]:
...

In [ ]:
grader.check("p1p1")

## Part 1.2) Choose a data type for each column



Notice the dtypes returned by `info()`. `object` here means string. These dtypes were chosen by pandas when it read the csv file, but they are not necessarily correct. We will now choose an appropriate data type for each column from these options: 

+ `"str"`: Strings
+ `"int"`: Integers
+ `"float"`: Floats
+ `"datetime"`: Date and time

Choose a data type for each column and save your selection to a Python dictionary called `chosen_dtypes`. The keys of this dictionary should be the six column headers. The values should be your chosen dtype for each column.

+ You can use `raw_data.columns` to get the list of column headers. 
+ Notice that you cannot cast something like `$12.50` to a float, because of the `$` symbol. You should therefore choose the `str` data type for `price` and we will take care of the `$` sign later. 

In [ ]:
chosen_dtypes =  ...

In [ ]:
grader.check("p1p2")

## Part 1.3) Repair data types

Having decided what type each column should be, we can now have pandas convert the data into those types. We will use the following pandas methods for doing this:

* `column.astype(dtype)`: `astype` operates on a column and transforms the data in that column into the given `dtype`. It works well for strings, but it only works for numbers when there are no missing values. In our case there are missing values, so for the numerical (int and float) columns we need to use the following method. 
* `pd.to_numeric(column,errors='coerce')`: This is a *package-level* method that transforms string columns into numerical columns. The argument `errors='coerce'` forces `to_numeric` to turn missing values into `NaN`s. 

    This method can detect whether the values are integers or floats, but only if there are no missing values. If there is even one missing value then it defaults to floats. To deal with this undesirable behavior, we need to explicitly cast the result of `to_numeric` to either an integer or a float. We do this by applying `astype` to the output of `to_numeric`.
    * For integers use: `pd.to_numeric(column,errors='coerce').astype('Int64')`
    * For floats use: `pd.to_numeric(column,errors='coerce').astype('Float64')`

* `pd.to_datetime(column, errors='coerce')`: This method is analogous to `to_numeric` for casting strings to date/time objects. 

**Instructions**

1. Create a copy of `raw_data` using its `copy()` method. Assign the result to the variable name `data`.
2. Cast each of the columns into the desired dtype using the methods listed above. To do this you should iterate through the items in the `chosen_dtypes` dictionary. The skeleton code below gives the syntax for doing this. 

**Hints**
* You can use `if` clauses inside the for loop to check the dtype and call the appropriate casting method. 

In [ ]:
data = ...

for col, dtype in chosen_dtypes.items():
    ...

# data.info()

In [ ]:
grader.check("p1p3")

## Part 1.4) Data inspection

Now that the types are properly set, we can proceed to check the data for possible problems. We will look for formatting inconsistencies, invalid values, missing entries, and column types that need repair.

**Instructions**

1. `store_location`: Use the `unique()` method to list the distinct location labels and look for problems caused by capitalization or extra spaces. 
2. `product_category`: Do the same as with `store_location`. 
3. `customer_age`: Use the `describe()` method to get information about the range of ages found in the data. 
4. `price`: Use the `unique()` method to inspect the entries of this column. You will see that the reason that this column failed to convert to numeric values was because of the presence of non-numeric charcaters.  
5. `customer_rating`: Use `isna()` and `sum()` to count the number of missing values in the customer ratings column. Save the integer number of missing customer ratings to the variable name `num_no_rating`.

Write a brief observation for each of items 1-4 in the dictionary called `observations`.

In [ ]:
observations = dict()

# 1. store_location
...
observations['store_location'] = "<ADD YOUR OBSERVATION HERE>"

# 2. product_category
...
observations['product_category'] =  "<ADD YOUR OBSERVATION HERE>"

# 3. customer_age
...
observations['customer_age'] =  "<ADD YOUR OBSERVATION HERE>"

# 4. price
...
observations['price'] =  "<ADD YOUR OBSERVATION HERE>"

# customer_rating
num_no_rating = ...

In [ ]:
grader.check("p1p4")

## Part 1.5) Data repair

In the previous part we identified a few problems with some of the columns. We will now fix those problems. 

**Instructions**

1. Create a copy of `data` using its `copy()` method. Assign the result to the variable name `data_clean`.
2. `store_location`: We saw some inconsistency in the use of upper and lower case letters in the city names. Also, some city names had leading or trailing white space. The city of Chicago, for example, shows up as `' Chicago'`,  `'CHICAGO'`, and `'Chicago'`. To fix this, apply these repairs:
    * Use the `column.str.strip()` method to remove leading and trailing whitespace from the column.
    * Use the `column.str.title()` method to convert the column entries to "titlecase".
3. `customer_age`: Use conditional indexing (`data_clean.loc[mask,:]`) to get rid of rows where the customer's age is less than 18 or greater than 120.
4. `price`: In order to successfully convert the prices into numbers, we need to remove all non-numerical characters. These are the dollar symbol `$` and the comma `,` (`.`, `+` and `-` are valid numerical characters). Apply the following repairs:
    * Use the `.str.replace` method to remove the `$` symbols and the commas (`,`) from the `price` columns.
    * Use `pd.to_numeric` to convert the prices into floats. 

In [ ]:
# 1. Make a copy of data
data_clean = ...

# 2. Fix store_location: remove whitespace and convert to title case
...

# 3. Fix customer_age: remove rows with customer_age outside of [18,120]
mask = ...
...

# 4. Fix price: remove '$' and ',', then convert to numeric type
...

In [ ]:
grader.check("p1p5")

## 1.6) Visualizations

Now that the data has been cleaned, it's time to make some plots. You are probably familiar with the Matplotlib package for plotting, and you can certainly use Matplotlib for visualizing data in a Pandas table. But Pandas also includes some plotting functionality attached to the DataFrame object, and it is sometimes easier to use that. Next we will see examples of both approaches. 

### 1.6.1) Sales per city

Create a bar plot showing the total sales in each city. 

1. Count how many records occur in each city and store the result as `counts`. This can be done by calling the `value_counts` method on the `store_location` column of `data_clean`. Store the result to `counts`. Check your result.  
2. Create the bar plot. You can use either of the following approaches:
    * Matplotlib approach: 
        ```python 
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.bar(x=counts.index,height=counts.values)
        ```
    * Pandas approach: 
        ```python 
        counts.plot(kind='bar',figsize=(8, 3))
        ```

Your plot should look something like this:

<center>
<img src="resources/barplot.png" alt="Alt text" width="500" />
</center>

In [ ]:
counts = ...

# using Matplotlib
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(x=counts.index,height=counts.values)

# using Pandas
# counts.plot(kind='bar',figsize=(8, 3))

In [ ]:
grader.check("p1p6p1")

### 1.6.2) Product price and customer age per product category

Suppose that we are interested in knowing something about how prices vary across different product categories (Accessories, Audio, Computers, and Mobile). We are also interested in the ages of customers purchasing products in the different categories. A "box plot" or "box and whisker plot" is a good visualization to use for this purpose. 

You can find a nice explanation of box plots [here](https://www.khanacademy.org/math/statistics-probability/summarizing-quantitative-data/box-whisker-plots/a/box-plot-review). Basically, each box in a box plot summarizes a set of values for a particular category (e.g. the prices of computers).

Make a plot with the following characteristics:
+ The figure has two rows and one column.
+ The top row shows a box plot of the customer ages for each product category. 
+ The bottom row shows a box plot of the product prices for each product category. 

You can use the `boxplot` method attached to the `data_clean` DataFrame to create the box plots. Your figure should look something like this:

<center>
<img src="resources/boxplot.png" alt="Alt text" width="300" />
</center>


In [ ]:
fig, axs = plt.subplots(figsize=(5, 6),nrows=2,sharex=True)
...

In [ ]:
grader.check("p1p6p2")

# Part 2) Simulating commute times


In the previous exercise we learned to deal with the imperfections of real data. Here we will learn to generate synthetic data from a probabilistic model. 

Our model is of the travel times experienced by commuters in their daily 12-mile trip. Each of the commuters uses a different mode of transportation.  
+ 15\% of them travel by **bike**.
+ 30\% use the **subway**.
+ 55\% drive a **car**. 

The travel times for each of these modes follow different normal distributions. 
+ The average travel time by **bike** is 50 minutes, with a 3-minute standard deviation.
+ The average travel time by **subway** is 30 minutes, with a 2-minute standard deviation.
+ The average travel time by **car** is 20 minutes, with a 5-minute standard deviation.

We will use symbols $M$ for the travel mode and $T$ for the travel time. The black box model is illustrated below. 

<center>
<img src="resources/blackbox.png" alt="Alt text" width="250" />
</center>

The travel mode $M$ follows a categorical discrete distribution:

$$p_M(m) \;=\; 
\begin{cases} 
0.15 & m=\text{bike} \\ 
0.30 & m=\text{subway} \\ 
0.55 & m=\text{car} 
\end{cases}$$

The normal distributions listed above give the probability of travel time conditioned on the mode:

$$T|M\!=\!m \; \sim \;
\begin{cases}
\mathcal{N}(50,3^2) & m=\text{bike} \\ 
\mathcal{N}(30,2^2) & m=\text{subway} \\ 
\mathcal{N}(20,5^2) & m=\text{car} 
\end{cases}
$$

The pdfs for $M$ and $T|M\!=\!m$ are shown below. 

<center>
<img src="resources/twopdfs.png" alt="Alt text" width="600" />
</center>

The marginal distribution of the travel time is then given by the following formula. 

\begin{align*} 
p_T(t) &= \sum_{m\in\Omega_M} p(t,m)  \\
&= \sum_{m\in\Omega_M} p_T(t|m) p_M(m) \\
&=  p_T(t|\text{bike}) p_M(\text{bike})  + p_T(t|\text{subway}) p_M(\text{subway})  + p_T(t|\text{car}) p_M(\text{car}) 
\end{align*}

This model for travel time is called a "mixture Gaussian model", since it consists of a combination of Gaussians. This is apparent in the plot of $p_T(t)$:

<center>
<img src="resources/pdfT.png" alt="Alt text" width="500" />
</center>


Our goal will be to create a function that generates samples of $T$. We will do so in three steps:

1. Create a function called `sample_M` that generates samples of $M$. 
2. Create a function called `sample_TgivenM` that generates samples of $T|M\!=\!m$.
3. Create a function called `sample_T` that uses `sample_M` and `sample_TgivenM` to generate samples of $T$.

## Part 2.1) Sampling $M$

Write a function called `sample_M(num_samples)` that takes a desired number of samples `num_samples` and returns a list of samples of $M$ of length `num_samples`. 

For example, if `num_samples=3`, then the function might return `['bike','car','car']`.

**Hints**
+ Use `rv_discrete` to create a discrete random variable $X$ with the same probabilities as $M$, but with sample space $\Omega_X=\{0,1,2\}$. Then map its samples (e.g. `[0,2,2]`) to samples of $M$ (e.g. `['bike','car','car']`).

In [ ]:
def sample_M(num_samples):

    # Sample space for M
    OmegaM = ['bike','subway','car']

    # Auxiliary random variable with same probabilities as M
    X = ...

    # Sample X
    samplesX = ...

    # Maps samples of X to values of M (doesn't have to be a single line of code)
    samplesM = ...
    
    return samplesM

In [ ]:
grader.check("p2p1")

## Part 2.2) Sampling $T|M\!=\!m$

Write a function called `sample_TgivenM(m_list)` that takes a list of values sampled from $M$ (e.g. `['bike','car','car']`) and returns a corresponding list of values sampled from $T|M\!=\!m$.

**Hint** 
+ The easiest way to do this is to iterate through `m_list` and for each value to sample the appropriate normal distribution. 

In [ ]:
def sample_TgivenM(m_array):

    # Allocate t_array using np.empty
    t_array = ...

    # iterate through m_array using `enumerate``
    for i, m in enumerate(m_array):

        # write cases for sampling a value from the appropriate normal distribution
        ...

        # store the value in t_array
        ...

    return t_array

In [ ]:
grader.check("p2p2")

## Part 2.3) Sampling $T$

Write a function called `sample_T(num_samples)` that takes a desired number of samples `num_samples` and returns a list of samples of $T$ of length `num_samples`. 

In [ ]:
def sample_T(num_samples):

    # 1. sample num_sample values of M
    ...

    # 2. sample corresponding values of T+M=m
    ...
    
    return ...

In [ ]:
grader.check("p2p3")

### Histogram of data generated with `sample_T`

The next cell uses your code to simulate 1000 samples of commuter travel times, and plots these samples in a histogram. Compare its shape to the pdf of $T$,

<img src="resources/pdfT.png" alt="Alt text" width="500" />


In [ ]:
fig, ax = plt.subplots(figsize=(5.5,3))
ax.hist(sample_T(1000),bins=30,rwidth=0.8,color='k',alpha=0.4)
ax.set_xlim(0,70)

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Please submit the .zip file to Gradescope.

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)